# Hadoop MapReduce Statistical Aggregates Processing
**Course:** 25MSDA3S01 – Big Data Analytics  
**Implementation:** Python + Hadoop Streaming

## Objective
Process numerical records and calculate maximum, arithmetic average, and exact distinct count. This notebook includes a local demonstration and the commands needed to run the same mapper/reducer through HDFS and Hadoop Streaming.

## Architecture
HDFS input → Mapper → Shuffle and sort → Reducer → HDFS output

The sample input is one number per line. The reducer also reports sum and record count to support verification.

## 1. Load the sample data

In [ ]:
from pathlib import Path

input_path = Path("dataset/numbers.txt")
numbers = [float(line.strip()) for line in input_path.read_text().splitlines() if line.strip()]
numbers

## 2. Manual calculation / expected answer

In [ ]:
total = sum(numbers)
record_count = len(numbers)
expected = {
    "Maximum": max(numbers),
    "Average": total / record_count,
    "Distinct_Count": len(set(numbers)),
    "Record_Count": record_count,
    "Sum": total,
}
expected

## 3. Mapper and reducer source
The standalone source files are `mapper.py` and `reducer.py`. The mapper emits `ALL<TAB>value`; Hadoop groups values with the same key and passes them to the reducer.

In [ ]:
from pathlib import Path

print("----- mapper.py -----")
print(Path("mapper.py").read_text())
print("----- reducer.py -----")
print(Path("reducer.py").read_text())

## 4. Run the mapper and reducer locally
This emulates the Hadoop Streaming pipeline on a small test file. Sorting by key is included to resemble the shuffle/sort stage.

In [ ]:
import subprocess
import sys

mapper_run = subprocess.run(
    [sys.executable, "mapper.py"],
    input=input_path.read_text(),
    text=True,
    capture_output=True,
    check=True,
)
mapper_output = mapper_run.stdout
print("Mapper output:")
print(mapper_output)

# Simulate Hadoop shuffle/sort by sorting intermediate key-value lines.
sorted_intermediate = "\n".join(sorted(mapper_output.splitlines())) + "\n"
reducer_run = subprocess.run(
    [sys.executable, "reducer.py"],
    input=sorted_intermediate,
    text=True,
    capture_output=True,
    check=True,
)
print("Reducer output:")
print(reducer_run.stdout)

## 5. Verify output against manual calculations

In [ ]:
actual = {}
for line in reducer_run.stdout.strip().splitlines():
    key, value = line.split("\t", 1)
    actual[key] = float(value)

assert actual["Maximum"] == expected["Maximum"]
assert actual["Average"] == expected["Average"]
assert actual["Distinct_Count"] == expected["Distinct_Count"]
assert actual["Record_Count"] == expected["Record_Count"]
assert actual["Sum"] == expected["Sum"]

print("PASS: MapReduce-style output matches the manual calculations.")
print("Expected:", expected)
print("Actual:", actual)

## 6. HDFS and Hadoop Streaming execution

Run these commands in a terminal inside the Hadoop environment, from the project directory. Make sure Python 3 and the Hadoop Streaming JAR are available to the task containers.

```bash
hdfs dfs -mkdir -p /user/student/statistics/input
hdfs dfs -put -f dataset/numbers.txt /user/student/statistics/input/
hdfs dfs -rm -r -f /user/student/statistics/output

hadoop jar /path/to/hadoop-streaming.jar \
  -D mapreduce.job.reduces=1 \
  -files mapper.py,reducer.py \
  -mapper "python3 mapper.py" \
  -reducer "python3 reducer.py" \
  -input /user/student/statistics/input/numbers.txt \
  -output /user/student/statistics/output

hdfs dfs -cat /user/student/statistics/output/part-00000
```

Replace `/path/to/hadoop-streaming.jar` with the actual JAR path in your cluster. If your Docker image uses a different Python executable, update `python3` accordingly.

## 7. Conclusion
For the included test data, the expected maximum is **80**, average is **35**, and distinct count is **7**. The local test confirms the mapper/reducer logic. For the final practical submission, also run the job on your Hadoop cluster and attach screenshots of the HDFS input, successful job completion, and HDFS output.

### Scalability consideration
This simple one-reducer implementation keeps distinct values in a Python set. It is suitable for a lab demonstration and exact distinct count, but a very large number of unique values can use substantial reducer memory. A production design should use a multi-stage aggregation or a distributed distinct-count approach.